# Investigating the Stability of Grad-CAM Explanations

Vanilla Grad-CAM on pretrained ResNet-50 and VGG-16, Oxford-IIIT Pet test set.
We apply small transformations that should not change the prediction and check whether the heatmap changes.

Metrics: Spearman rank correlation, SSIM, IoU between thresholded heatmaps, IoU with ground-truth mask, pointing game.

In [ ]:
import os
import pandas as pd

from src import config
from src.dataset import load_pet_dataset
from run_experiments import main as run_all_experiments
from src import visualize as vis

pd.set_option("display.max_rows", 100)
print("Device:", config.DEVICE)
print("Models:", config.MODELS, "| Images:", config.NUM_IMAGES)

## 1. Run experiments
Skip this cell if `results/*_results.csv` already exist (e.g. created with `python run_experiments.py`).

In [ ]:
if not all(os.path.exists(os.path.join(config.RESULTS_DIR, f"{m}_results.csv")) for m in config.MODELS):
    run_all_experiments()

In [ ]:
dataset = load_pet_dataset()
all_results = {m: vis.load_results(m) for m in config.MODELS}

## 2. Summary tables (mean and std across images)
Only cases where the prediction stayed the same. `identity` is the sanity check (should be 1.0), `random_pair` is the lower reference (heatmaps of two different images), `center_baseline` is a centre Gaussian map for localisation.

In [ ]:
vis.summary_table(all_results["resnet50"])

In [ ]:
vis.summary_table(all_results["vgg16"])

## 3. RQ1: How sensitive is Grad-CAM to each transformation?

In [ ]:
for model_name, results in all_results.items():
    vis.plot_stability_bars(results, model_name)

## 4. RQ2: Sensitivity analysis over transformation strength

In [ ]:
for model_name, results in all_results.items():
    vis.plot_strength_curves(results, model_name)
    vis.plot_spearman_boxplot(results, model_name)

## 5. Localisation against ground-truth masks (IoU and pointing game)

In [ ]:
for model_name, results in all_results.items():
    vis.plot_localization(results, model_name)

## 6. Architecture comparison: ResNet-50 vs VGG-16

In [ ]:
vis.plot_model_comparison(all_results)

## 7. Qualitative examples

In [ ]:
example_index = int(all_results["resnet50"]["image_index"].iloc[0])
for model_name in config.MODELS:
    vis.show_heatmaps(dataset, model_name, example_index)

## 8. Failure cases: heatmap changed a lot but prediction did not

In [ ]:
for model_name, results in all_results.items():
    worst = vis.worst_cases(results, n=3)
    display(worst)
    for _, row in worst.iterrows():
        settings = [("identity", 0), (row["transform"], row["strength"])]
        vis.show_heatmaps(dataset, model_name, int(row["image_index"]), settings)